# Measure analysis

##### Imports and params

In [ ]:
from __future__ import annotations

import pandas as pd

from pathlib import Path
import matplotlib.pyplot as plt
import openalea.rsml as rsml

In [ ]:
plt.rcParams.update({
"figure.figsize": (18, 10),
"axes.grid": True,
"grid.linestyle": ":",
"grid.alpha": 0.5,
"axes.titlesize": 14,
"axes.labelsize": 12,
"legend.fontsize": 10,
})

In [ ]:
MORPHO_RESULTS = Path("/home/loai/Documents/code/RSMLExtraction/Measures/measureresults_per_plant.csv").expanduser()
GT_MORPHO_RESULTS = Path("/home/loai/Documents/code/RSMLExtraction/RSA_reconstruction/Method/ChronoRoot/Ground_truth_root.csv").expanduser()
PATH_2_LOGS = Path("/home/loai/Documents/code/RSMLExtraction/Measures/all_events_combined.csv").expanduser()

## Loading database

In [ ]:
df_base = pd.read_csv(MORPHO_RESULTS)
df_base.head()

In [ ]:
    # divide model into three columns, model_name, loss_name and epoch
df_base['model_name'] = df_base['model'].str.split('_').str[0]
pos_number_in_model = df_base['model'].str.split('_').apply(lambda x: next((i for i, part in enumerate(x) if part.isdigit()), -1))


df_base['loss_name'] = df_base.apply(
	lambda row: '_'.join(row['model'].split('_')[1:pos_number_in_model[row.name]])
	if pos_number_in_model[row.name] != -1
	else '_'.join(row['model'].split('_')[1:]),
	axis=1
)

def extract_epoch(row):
	pos = pos_number_in_model[row.name]
	if pos != -1:
		part = row['model'].split('_')[pos]
		return int(part.replace('epoch', '')) if 'epoch' in part else int(part)
	else:
		return -1

df_base['epoch'] = df_base.apply(extract_epoch, axis=1)

# remove _number from loss_name
df_base['loss_name'] = df_base['loss_name'].str.replace(r'_\d+', '', regex=True)
df_base.head()


In [ ]:
# get all boxes in the df_base
boxes = df_base['box'].unique()
path_2_rsmls = "/home/loai/Images/DataTest/UC1/UC1_OLD/"
# foe each box, path_2_rsmls + box + "/61_graph_expertized.rsml"
for box in boxes:
    rsml_path = Path(path_2_rsmls) / f"{box}/61_graph_expertized.rsml"
    graph = rsml.rsml2mtg(str(rsml_path))
    metadata = graph.graph_properties().get('metadata', {})
        
    obs_hours = [float(h) for h in metadata.get('observation-hours', "").split(',') if h]
    pixel_size = float(metadata.get('pixel-size', 76.0))
    
    mask = df_base['box'] == box
    df_base.loc[mask, 'pixel_size'] = pixel_size * 1e-3 # convert to mm
    
    def map_time(idx):
        try:
            return obs_hours[int(idx) - 1]
        except (IndexError, ValueError):
            return None
    df_base.loc[mask, 'time_hours'] = df_base.loc[mask, 'time'].apply(map_time)
df_base.head()

In [ ]:
combination_counts = df_base.groupby(['model_name', 'loss_name']).size().reset_index(name='counts')

combination_counts = combination_counts.groupby(['model_name', 'loss_name']).agg({'counts': 'sum'}).reset_index()
print(combination_counts)

In [ ]:
df_base = df_base.drop(columns=['model', 'split'])

# order by model_name, loss_name, epoch, box and time
df_base = df_base.sort_values(by=['model_name', 'loss_name', 'epoch', 'box', 'time', 'time_hours', 'pixel_size'])

# pivot on source "source" and value is "value" column 
df_base = df_base.pivot_table(index=['model_name', 'loss_name', 'epoch', 'box', 'time', 'time_hours', 'root_ids', 'metric', 'pixel_size'], columns='source', values='value').reset_index()

# for col not in 'model_name', 'loss_name', 'epoch', 'box', 'time', 'time_hours', 'root_ids', 'metric', 'pixel_size', multiply by pixel_size
metadata_cols = ['model_name', 'loss_name', 'epoch', 'box', 'time', 'time_hours', 'root_ids', 'metric', 'pixel_size']
for col in df_base.columns:
    if col not in metadata_cols:
        df_base[col] = df_base[col] * df_base['pixel_size']
    
df_base.drop(columns=['pixel_size'], inplace=True)

# remove rows where metric is Intercept_curve_Area
df_base = df_base[df_base['metric'] != 'Intercept_curve_Area']
# remove loss_name = 'bce_dice' 
df_base = df_base[df_base['loss_name'] != 'bce_dice']
        

df_base.head()


In [ ]:
combination_counts = df_base.groupby(['model_name', 'loss_name']).size().reset_index(name='counts')
print(combination_counts)

In [ ]:
# metrcis should be column and not rows, so we need to pivot the table
df_plant = df_base.pivot(index=['model_name', 'loss_name', 'epoch', 'box', 'time', 'time_hours', 'root_ids'], columns="metric", values="Prediction").reset_index()
df_plant.head()

In [ ]:
df_plant_gt = df_base.pivot(index=['model_name', 'loss_name', 'epoch', 'box', 'time', 'time_hours', 'root_ids'], columns="metric", values="expertized").reset_index()
df_plant_bgt = df_base.pivot(index=['model_name', 'loss_name', 'epoch', 'box', 'time', 'time_hours', 'root_ids'], columns="metric", values="before_expertized").reset_index()

df_plant_gt.head()

In [ ]:
def process_plant_kinematics(df):
    metadata_cols = ['model_name', 'loss_name', 'epoch', 'box', 'time', 'time_hours', 'root_ids']
    no_speed_cols = ['NumberOfOrgans', 'NumberOfLateralRoots', 'Intercept_curve_Area']
    metrics = [c for c in df.columns if c not in metadata_cols or c in no_speed_cols]
    
    df = df.sort_values(by=['model_name', 'loss_name', 'epoch', 'box', 'root_ids', 'time_hours'])
    
    grouper = df.groupby(['model_name', 'loss_name', 'epoch', 'box', 'root_ids'])
    
    for metric in metrics:
        speed_col = f"{metric}_speed"
        df[speed_col] = grouper[metric].diff() / grouper['time_hours'].diff()
        
        accel_col = f"{metric}_acceleration"
        df[accel_col] = grouper[speed_col].diff() / grouper['time_hours'].diff()
        
    return df

# Apply to your three dataframes
df_plant = process_plant_kinematics(df_plant)
df_plant_gt = process_plant_kinematics(df_plant_gt)
df_plant_bgt = process_plant_kinematics(df_plant_bgt)

# Preview results for one metric (e.g., 'length')
df_plant

### Logs

In [ ]:
cols_to_keep = ['step', 'value', 'folder', 'metric']
df_logs_tb = pd.read_csv(PATH_2_LOGS, usecols=lambda c: c in cols_to_keep)

# create new columns for model name and loss name
df_logs_tb['model_name'] = df_logs_tb['folder'].str.split('/').str[9].str.split('_').str[0]
df_logs_tb['loss_name'] = df_logs_tb['folder'].str.split('/').str[9].str.split('_').str[1:].str.join('_')

df_logs_tb = df_logs_tb.rename(columns={"step": "epoch", "value": "model_metric_value"})

df_logs_tb = df_logs_tb[df_logs_tb['metric'] != 'batch_loss']
# drop learning rate
df_logs_tb = df_logs_tb[df_logs_tb['metric'] != 'learning_rate']

df_logs_tb = df_logs_tb.drop(columns=['folder'])

print("Available loss names:", df_logs_tb['loss_name'].unique().tolist())
df_logs_tb.head()


In [ ]:
# all model_name, loss_name and metric + model_metric_value at last epoch
df_logs_tb_last_epoch = df_logs_tb.groupby(['model_name', 'loss_name', 'metric']).last().reset_index()
df_logs_tb_last_epoch

## Prepare diff datasets

In [ ]:
# all columns that are not model_name, loss_name, epoch, box and time are metrics
metric_columns = [col for col in df_plant.columns if col not in ['model_name', 'loss_name', 'epoch', "box", "time", "root_ids"]]
df_diff = df_plant_gt.copy()

for metric in metric_columns:
    df_diff[f"{metric}_signed_err"] = df_plant[metric] - df_plant_gt[metric]
    
    df_diff[f"{metric}_abs_err"] = df_diff[f"{metric}_signed_err"].abs()
    
    floatgt = abs(df_plant_gt[metric].astype(float).replace(0, pd.NA)) # 
    df_diff[f'{metric}_rel_err'] = df_diff[f"{metric}_abs_err"] / (floatgt + 1e-9)
    
# drop TimeStep_signed_err, TimeStep_abs_err and TimeStep_rel_err if they exist
df_diff = df_diff.drop(columns=metric_columns)

#df_diff.dropna(inplace=True)

print("Available loss names:", df_diff['loss_name'].unique().tolist())

df_diff.head()

## Merge with logs

In [ ]:
subset_keys = ['epoch', 'loss_name', 'model_name', 'metric']

# remove metric = train/batch_loss
df_logs_tb = df_logs_tb[~df_logs_tb['metric'].isin(['train/batch_loss', 'train/lr'])]

# metric named : val/val_loss_BCE, val/val_loss_BCEDiceLoss, val/val_loss_CLDice, val/val_loss_CLDice_Dice and val/val_loss_DiceLoss
# Combine all these columns into one column named val/val_loss, and drop the original columns

for vallos in ['val/val_loss_BCE', 'val/val_loss_BCEDiceLoss', 'val/val_loss_CLDice', 'val/val_loss_CLDice_Dice', 'val/val_loss_DiceLoss']:
    if vallos in df_logs_tb['metric'].unique():
        df_logs_tb.loc[df_logs_tb['metric'] == vallos, 'metric'] = 'val/val_loss'
    else:
        print(f"Metric {vallos} not found in logs. Setting val/val_loss to N/A for this metric.")

In [ ]:
duplicates_mask = df_logs_tb.duplicated(subset=subset_keys, keep=False)
df_duplicates = df_logs_tb[duplicates_mask]

df_duplicates = df_duplicates.sort_values(by=subset_keys)

print(f"Nombre total de lignes dupliquées : {len(df_duplicates)}")
print("Aperçu des doublons :")
df_duplicates

In [ ]:
last_epochs = df_logs_tb.groupby(['model_name', 'loss_name'])['epoch'].max().reset_index()
last_epoch_dict = {(row['model_name'], row['loss_name']): row['epoch'] for _, row in last_epochs.iterrows()}

print(last_epoch_dict)

In [ ]:
def replace_epoch(row):
    if row['epoch'] == -1:
        return last_epoch_dict.get((row['model_name'], row['loss_name']), -1)
    
    return row['epoch']

df_diff['epoch'] = df_diff.apply(replace_epoch, axis=1)
df_diff.head()


In [ ]:
df_logs_pivoted = df_logs_tb.pivot_table(
    index=['epoch', 'loss_name', 'model_name'], 
    columns='metric', 
    values='model_metric_value',
    aggfunc='first'  
).reset_index()

# val/val_loss_BCE, val/val_loss_CLDice_Dice and val/val_loss_DiceLoss should have missing values -> combine them into one column val/val_loss
#df_logs_pivoted['val/val_loss'] = df_logs_pivoted[['val/val_loss_BCE', 'val/val_loss_CLDice_Dice', 'val/val_loss_DiceLoss']].bfill(axis=1).iloc[:, 0]
#df_logs_pivoted = df_logs_pivoted.drop(columns=['val/val_loss_BCE', 'val/val_loss_CLDice_Dice', 'val/val_loss_DiceLoss'])

print("Available loss names:", df_logs_pivoted['loss_name'].unique().tolist())

df_logs_pivoted

In [ ]:
df_diff_log = pd.merge(
    df_diff,
    df_logs_pivoted, 
    on=['epoch', 'loss_name', 'model_name'],
    how='left'
)

#df_diff_log = df_diff_log.dropna()

# print number of lines for every model_name x loss_name combination
combination_counts = df_diff_log.groupby(['model_name', 'loss_name']).size().reset_index(name='counts')
print(combination_counts)


df_diff_log.head()

## Visualize raw data

### Utils

In [ ]:
import ipywidgets as widgets
from IPython.display import display

In [ ]:
# List all loss names, model names, and metrics
loss_names = df_diff_log['loss_name'].unique().tolist()
model_names = df_diff_log['model_name'].unique().tolist()
col_box_name = df_diff_log["box"].unique().tolist()
col_plant_number = df_diff_log["root_ids"].unique().tolist()
col_time_hours = df_diff_log['time'].unique().tolist()
col_time_hours.sort()
col_epoch = df_diff_log["epoch"].unique().tolist()
col_epoch.sort()
img_metrics = df_logs_tb['metric'].unique().tolist()
plant_errors = df_diff_log.columns.tolist()
# remove everything that is not a measure
for col in ['epoch', 'loss_name', 'model_name', 'box',  'root_ids', 'TimeStep', 'FileName', 'time', 'Acquisition Time'] + img_metrics:
    if col in plant_errors:
        plant_errors.remove(col)
print("Available loss names:", loss_names)
print("Available model names:", model_names)
print("Available box names:", col_box_name)
print("Available plant numbers:", col_plant_number)
print("Available epochs:", col_epoch)
print("Available image metrics:", img_metrics)
print("Available plant errors:", plant_errors)


### Visu

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt

# --- 1. Define Widgets ---

w_model = widgets.Dropdown(
    options=sorted(df_diff_log['model_name'].unique()), 
    description='Model:'
)

w_loss = widgets.Dropdown(
    options=sorted(df_diff_log['loss_name'].unique()), 
    description='Loss:'
)

w_box = widgets.Dropdown(
    options=sorted(df_diff_log['box'].unique()), 
    description='Box:'
)

# Initialize with a placeholder to prevent "nonempty" error on startup
w_epoch = widgets.SelectionSlider(
    options=['N/A'], 
    description='Epoch:',
    continuous_update=False,
    disabled=True 
)

w_root_id = widgets.Dropdown(
    options=[], 
    description='Root ID:'
)

w_measure = widgets.Dropdown(
    options=plant_errors, 
    description='Measure:'
)

def update_dependent_options(*_):
    curr_model = w_model.value
    curr_loss = w_loss.value
    curr_box = w_box.value
    
    # Filter data
    subset = df_diff_log[
        (df_diff_log['model_name'] == curr_model) & 
        (df_diff_log['loss_name'] == curr_loss) & 
        (df_diff_log['box'] == curr_box)
    ]
    
    # --- Update Epochs (Fix for TraitError) ---
    available_epochs = sorted(subset['epoch'].unique())
    
    if len(available_epochs) > 0:
        w_epoch.options = available_epochs
        w_epoch.disabled = False
        # Preserve selection if valid
        if w_epoch.value not in available_epochs:
            w_epoch.value = available_epochs[-1]
    else:
        # Fallback if no data: set a dummy option
        w_epoch.options = ['N/A'] 
        w_epoch.disabled = True

    available_roots = sorted(subset['root_ids'].unique())
    
    if len(available_roots) > 0:
        w_root_id.options = available_roots
        w_root_id.disabled = False
        if w_root_id.value not in available_roots:
            w_root_id.value = available_roots[0]
    else:
        w_root_id.options = ['N/A']
        w_root_id.disabled = True

w_model.observe(update_dependent_options, names='value')
w_loss.observe(update_dependent_options, names='value')
w_box.observe(update_dependent_options, names='value')

update_dependent_options()

def plot_logic(model_name, loss_name, epoch, box, root_id, measure):
    if epoch == 'N/A' or root_id == 'N/A':
        plt.figure(figsize=(14, 6))
        plt.text(0.5, 0.5, "No data available for this selection", 
                 ha='center', va='center', fontsize=14)
        plt.tight_layout()
        plt.show()
        return

    df_filtered = df_diff_log[
        (df_diff_log['model_name'] == model_name) &
        (df_diff_log['loss_name'] == loss_name) &
        (df_diff_log['epoch'] == epoch) &
        (df_diff_log['box'] == box) &
        (df_diff_log['root_ids'] == root_id)
    ]
    
    df_filtered = df_filtered.sort_values('time').copy()
    # remove rows where measure is NaN
    df_filtered = df_filtered[~df_filtered[measure].isna()]

    plt.figure(figsize=(14, 6))
    
    if df_filtered.empty:
        plt.text(0.5, 0.5, "No data found", ha='center', va='center', fontsize=14)
    else:
        plt.plot(df_filtered['time'], df_filtered[measure], 
                 marker='o', color='teal', label=measure, 
                 linewidth=2, markersize=4, alpha=0.8)
        
        plt.title(f'Analysis: {measure}\n({model_name} | {loss_name} | Epoch {epoch} | Box {box} | Root {root_id})')
        plt.xlabel('Time (hours)')
        plt.ylabel(measure)
        plt.legend()
        plt.grid(True, linestyle=':', alpha=0.6)
    
    plt.tight_layout()
    plt.show()

ui = widgets.VBox([
    widgets.HBox([w_model, w_loss, w_epoch]),
    widgets.HBox([w_box, w_root_id]),
    widgets.HBox([w_measure])
])

out = widgets.interactive_output(plot_logic, {
    'model_name': w_model,
    'loss_name': w_loss,
    'epoch': w_epoch,
    'box': w_box,
    'root_id': w_root_id,
    'measure': w_measure
})

display(ui, out)

## Per epoch visualization

In [ ]:
import pandas as pd
from IPython.display import display

target_metrics = [m for m in plant_errors if 'rel_err' in m ]

# Remplacer -1 par l'epoch max pour chaque combinaison model_name x loss_name
last_epochs = df_diff_log.groupby(['model_name', 'loss_name'])['epoch'].max().reset_index()
last_epoch_dict = {(row['model_name'], row['loss_name']): row['epoch'] for _, row in last_epochs.iterrows()}
df_diff_log['epoch'] = df_diff_log.apply(lambda row: last_epoch_dict.get((row['model_name'], row['loss_name']), row['epoch']) if row['epoch'] == -1 else row['epoch'], axis=1)

df_grouped = df_diff_log[df_diff_log['epoch'] == df_diff_log.groupby(['model_name', 'loss_name'])['epoch'].transform('max')].groupby(['model_name', 'loss_name'])[target_metrics].mean()
best_configs = []

for metric in target_metrics:
    min_val = df_grouped[metric].min()
    winners = df_grouped[df_grouped[metric] <= min_val + 1e-4]
    for (model, loss), row in winners.iterrows():
        best_configs.append({
            'Metric': metric,
            'epoch': df_diff_log[df_diff_log['model_name'] == model][df_diff_log['loss_name'] == loss]['epoch'].max(),
            'Best Model': model,
            'Best Loss': loss,
            'Min Mean Error': row[metric]
        })

df_best = pd.DataFrame(best_configs)

display(df_best.style.background_gradient(subset=['Min Mean Error'], cmap='Greens_r'))

In [ ]:
import pandas as pd
from IPython.display import display

target_metrics = [m for m in plant_errors if 'rel_err' in m ]

last_epochs = df_diff_log.groupby(['model_name', 'loss_name'])['epoch'].max().reset_index()
last_epoch_dict = {(row['model_name'], row['loss_name']): row['epoch'] for _, row in last_epochs.iterrows()}
df_diff_log['epoch'] = df_diff_log.apply(lambda row: last_epoch_dict.get((row['model_name'], row['loss_name']), row['epoch']) if row['epoch'] == -1 else row['epoch'], axis=1)

df_grouped = df_diff_log[df_diff_log['epoch'] == df_diff_log.groupby(['model_name', 'loss_name'])['epoch'].transform('max')].groupby(['model_name', 'loss_name'])[target_metrics].mean() # meean over all measures for each model_name x loss_name combination at last epoch
best_configs = []

for metric in target_metrics:
    max_val = df_grouped[metric].max()
    winners = df_grouped[df_grouped[metric] >= max_val - 1e-4]
    for (model, loss), row in winners.iterrows():
        best_configs.append({
            'Metric': metric,
            'epoch': df_diff_log[df_diff_log['model_name'] == model][df_diff_log['loss_name'] == loss]['epoch'].max(),
            'Best Model': model,
            'Best Loss': loss,
            'Max Mean Error': row[metric]
        })

df_best = pd.DataFrame(best_configs)

display(df_best.style.background_gradient(subset=['Max Mean Error'], cmap='Reds_r'))

## Correlation tables

In [ ]:
from scipy.stats import spearmanr, pearsonr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plant_errors = ['NumberOfOrgans_signed_err', 'NumberOfOrgans_abs_err', 'NumberOfOrgans_rel_err', 'TotalRootLength_signed_err', 'TotalRootLength_abs_err', 'TotalRootLength_rel_err', 'TotalRootLength_speed_signed_err', 'TotalRootLength_speed_rel_err', 'TotalRootLength_acceleration_signed_err', 'TotalRootLength_acceleration_rel_err', "TotalRootLength_speed_abs_err", "TotalRootLength_acceleration_abs_err"] # 'Convex_Area_Hull_signed_err', 'Convex_Area_Hull_abs_err', 'Convex_Area_Hull_rel_err', 'RootDensity_signed_err', 'RootDensity_abs_err', 'RootDensity_rel_err'

loss_metrics = ['train/epoch_loss', 'val/Dice', 'val/CLDice','val/FocalLoss', 'val/val_loss']
img_metrics =  ['val/Precision', 'val/Recall', 'val/F1Score', 'val/MeanIoU', 'val/HausdorffDistance', 'val/AverageCenterlineDistance', 'val/CenterlineF1'] #, 'val/Betti0JaccardRatioGPU', 'val/Betti0RelativeErrorGPU', 'val/Betti0VariationIndexGPU', 'val/Betti1JaccardRatioGPU', 'val/Betti1RelativeErrorGPU', 'val/Betti1VariationIndexGPU']

scores_img_metrics = ['val/F1Score', 'val/MeanIoU', 'val/Precision', 'val/Recall', 'val/CenterlineF1']

plant_abs_err =  [col for col in plant_errors if 'abs_err' in col]
plant_rel_err = [col for col in plant_errors if 'rel_err' in col]
plant_signed_err = [col for col in plant_errors if 'signed_err' in col]

In [ ]:
# mean value over all epochs 
df_diff_log_epoch_mean = df_diff_log.copy()
df_diff_log_epoch_mean.drop(columns=['root_ids', 'box', 'time'], inplace=True)
df_diff_log_epoch_mean = df_diff_log_epoch_mean.groupby(['model_name', 'loss_name', 'epoch']).mean().reset_index() # does not average segmentation metrics because for 1 model x loss x epoch = 1 segmentation value but multiple plants values
df_diff_log_epoch_mean

In [ ]:
import numpy as np
from scipy.stats import spearmanr

rho_rows, p_rows = [], []
for img_met in img_metrics:
    rho_row, p_row = {}, {}
    for plant_err in plant_rel_err:
        #valid = df_diff_log[[img_met, plant_err]].dropna().copy()
        valid = df_diff_log_epoch_mean[[img_met, plant_err]].dropna().copy()
        if len(valid) < 3:
            rho, p = np.nan, np.nan
        else:
            if img_met in scores_img_metrics:
                valid[img_met] = - valid[img_met]
            rho, p = spearmanr(valid[img_met], valid[plant_err])
        rho_row[plant_err] = float(rho) if np.isfinite(rho) else np.nan
        p_row[plant_err]   = float(p) if np.isfinite(p) else np.nan
    rho_rows.append(pd.Series(rho_row, name=img_met))
    p_rows.append(pd.Series(p_row, name=img_met))

CorrSpearman = pd.DataFrame(rho_rows)
Pvalues      = pd.DataFrame(p_rows)

print("Corrélation (ρ Spearman):")
display(CorrSpearman.round(3))

print("p-values:")
display(Pvalues.applymap(lambda x: round(x, 3) if pd.notna(x) else x))

# --- Heatmap simple (matplotlib) centrée sur 0 ---
plt.figure(figsize=(1.2*len(plant_rel_err)+3, 0.45*len(img_metrics)+3))
im = plt.imshow(CorrSpearman.values, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, fraction=0.046, pad=0.04, label="Spearman ρ (img_metric vs plant_error)")
plt.yticks(range(len(img_metrics)), img_metrics)
plt.xticks(range(len(plant_rel_err)), plant_rel_err, rotation=45, ha="right")
plt.title("Correlation table (Spearman) between Segmentation metrics and Plant errors", pad=20)

# Annotations: ρ
for i in range(len(img_metrics)):
    for j in range(len(plant_rel_err)):
        rho = CorrSpearman.iat[i, j]
        p = Pvalues.iat[i, j]
        txt = "" if np.isnan(rho) else f"{rho:.3f}\n(p={p:.3f})"
        plt.text(j, i, txt, ha='center', va='center', fontsize=8, color="black")
plt.tight_layout()
plt.savefig("corr_img_to_plant_error_spearman.png", transparent=True, dpi=600)
plt.show()

# Option: export CSV
#CorrSpearman.to_csv("corr_img_to_plant_error_spearman.csv")
#Pvalues.to_csv("corr_img_to_plant_error_spearman_pvalues.csv")

In [ ]:
import numpy as np
from scipy.stats import pearsonr

rho_rows, p_rows = [], []
for img_met in img_metrics:
    rho_row, p_row = {}, {}
    for plant_err in plant_abs_err:
        #valid = df_diff_log[[img_met, plant_err]].dropna().copy()
        valid = df_diff_log_epoch_mean[[img_met, plant_err]].dropna().copy()
        if len(valid) < 3:
            rho, p = np.nan, np.nan
        else:
            if img_met in scores_img_metrics:
                valid[img_met] = - valid[img_met]
            rho, p = pearsonr(valid[img_met], valid[plant_err])
        rho_row[plant_err] = float(rho) if np.isfinite(rho) else np.nan
        p_row[plant_err]   = float(p) if np.isfinite(p) else np.nan
    rho_rows.append(pd.Series(rho_row, name=img_met))
    p_rows.append(pd.Series(p_row, name=img_met))

CorrSpearman = pd.DataFrame(rho_rows)
Pvalues      = pd.DataFrame(p_rows)

print("Corrélation (ρ Pearson):")
display(CorrSpearman.round(3))

print("p-values:")
display(Pvalues.applymap(lambda x: round(x, 3) if pd.notna(x) else x))

# --- Heatmap simple (matplotlib) centrée sur 0 ---
plt.figure(figsize=(1.2*len(plant_abs_err)+3, 0.45*len(img_metrics)+3))
im = plt.imshow(CorrSpearman.values, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, fraction=0.046, pad=0.04, label="Pearson r (img_metric vs plant_error)")
plt.yticks(range(len(img_metrics)), img_metrics)
plt.xticks(range(len(plant_abs_err)), plant_abs_err, rotation=45, ha="right")
plt.title("Correlation table (Pearson) between Segmentation metrics and Plant errors", pad=20)

# Annotations: ρ
for i in range(len(img_metrics)):
    for j in range(len(plant_abs_err)):
        rho = CorrSpearman.iat[i, j]
        p = Pvalues.iat[i, j]
        txt = "" if np.isnan(rho) else f"{rho:.3f}\n(p={p:.3f})"
        plt.text(j, i, txt, ha='center', va='center', fontsize=8, color="black")
plt.tight_layout()
plt.savefig("corr_img_to_plant_error_pearson.png", transparent=True, dpi=600)
plt.show()

# Option: export CSV
#CorrSpearman.to_csv("corr_img_to_plant_error_pearson.csv")
#Pvalues.to_csv("corr_img_to_plant_error_pearson_pvalues.csv")

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr

def phase_scramble(series, n_surrogates=1000):
    """
    Génère des 'surrogates' (substituts) qui préservent l'autocorrélation (spectre de puissance)
    mais détruisent la phase (relation temporelle spécifique).
    """
    series = np.array(series)
    n = len(series)
    
    # FFT
    f_series = np.fft.rfft(series)
    
    # Génération des phases aléatoires pour chaque surrogate
    # On ne touche pas à la composante DC (f=0) ni à Nyquist si n est pair
    random_phases = np.random.uniform(0, 2*np.pi, (n_surrogates, len(f_series)))
    random_phases[:, 0] = 0 # DC component fixée
    if n % 2 == 0:
        random_phases[:, -1] = 0 # Nyquist fixée si pair
        
    # Ajout des phases aléatoires (rotation dans le plan complexe)
    # f_surrogate = amplitude * exp(i * (phase_originale + phase_aleatoire))
    f_surrogates = f_series * np.exp(1j * random_phases)
    
    # Inverse FFT pour revenir au temporel
    surrogates = np.fft.irfft(f_surrogates, n=n, axis=1)
    
    return surrogates

def ebisuzaki_correlation_test(x, y, n_surrogates=1000, metric='pearson'):
    """
    Calcule la p-value corrigée pour l'autocorrélation via Phase Scrambling.
    """
    # Nettoyage des NaNs
    mask = ~np.isnan(x) & ~np.isnan(y)
    x_clean, y_clean = x[mask], y[mask]
    
    if len(x_clean) < 3:
        return np.nan, np.nan

    # 1. Corrélation observée
    if metric == 'pearson':
        r_obs, _ = pearsonr(x_clean, y_clean)
    elif metric == 'spearman':
        r_obs, _ = spearmanr(x_clean, y_clean)
        
    # 2. Génération des surrogates pour X (Phase Scrambling)
    x_surrogates = phase_scramble(x_clean, n_surrogates)
    
    # 3. Calcul des corrélations nulles
    r_null = []
    for x_surr in x_surrogates:
        if metric == 'pearson':
            r, _ = pearsonr(x_surr, y_clean)
        elif metric == 'spearman':
            r, _ = spearmanr(x_surr, y_clean)
        r_null.append(r)
    
    r_null = np.array(r_null)
    
    # 4. Calcul de la p-value (Test bilatéral)
    # Proportion des surrogates qui ont une corrélation absolue plus forte que l'observée
    p_value = np.mean(np.abs(r_null) >= np.abs(r_obs))
    
    return r_obs, p_value

# --- Exemple d'utilisation dans votre boucle ---
# rho, p = ebisuzaki_correlation_test(valid[img_met], valid[plant_err], n_surrogates=2000)

In [ ]:
import numpy as np
from scipy.stats import pearsonr

rho_rows, p_rows = [], []
for img_met in img_metrics:
    rho_row, p_row = {}, {}
    for plant_err in plant_rel_err:
        # use ebisuzaki_correlation_test instead of pearsonr
        valid = df_diff_log_epoch_mean[[img_met, plant_err]].dropna().copy()
        
        if len(valid) < 3:
            rho, p = np.nan, np.nan
        else:
            if img_met in scores_img_metrics:
                valid[img_met] = - valid[img_met]
            rho, p = ebisuzaki_correlation_test(valid[img_met], valid[plant_err], n_surrogates=2000, metric='pearson')
        rho_row[plant_err] = float(rho) if np.isfinite(rho) else np.nan
        p_row[plant_err]   = float(p) if np.isfinite(p) else np.nan
    rho_rows.append(pd.Series(rho_row, name=img_met))
    p_rows.append(pd.Series(p_row, name=img_met))

CorrPearson = pd.DataFrame(rho_rows)
Pvalues      = pd.DataFrame(p_rows)

print("Corrélation (ρ Pearson):")
display(CorrPearson.round(3))

print("p-values:")
display(Pvalues.applymap(lambda x: round(x, 3) if pd.notna(x) else x))

# --- Heatmap simple (matplotlib) centrée sur 0 ---
plt.figure(figsize=(1.2*len(plant_rel_err)+3, 0.45*len(img_metrics)+3))
im = plt.imshow(CorrPearson.values, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, fraction=0.046, pad=0.04, label="Pearson r (img_metric vs plant_error)")
plt.yticks(range(len(img_metrics)), img_metrics)
plt.xticks(range(len(plant_rel_err)), plant_rel_err, rotation=45, ha="right")
plt.title("Correlation table (Pearson) between Segmentation metrics and Plant errors", pad=20)

# Annotations: r
for i in range(len(img_metrics)):
    for j in range(len(plant_rel_err)):
        rho = CorrPearson.iat[i, j]
        p = Pvalues.iat[i, j]
        txt = "" if np.isnan(rho) else f"{rho:.3f}\n(p={p:.3f})"
        plt.text(j, i, txt, ha='center', va='center', fontsize=8, color="black")
plt.tight_layout()
plt.savefig("corr_img_to_plant_error_pearson.png", transparent=True, dpi=600)
plt.show()

# Option: export CSV
#CorrPearson.to_csv("corr_img_to_plant_error_pearson.csv")
#Pvalues.to_csv("corr_img_to_plant_error_pearson_pvalues.csv")

In [ ]:
import numpy as np

rho_rows, p_rows = [], []
for img_met in img_metrics:
    rho_row, p_row = {}, {}
    for plant_err in plant_rel_err:
        # use ebisuzaki_correlation_test instead of pearsonr
        valid = df_diff_log_epoch_mean[[img_met, plant_err]].dropna().copy()
        
        if len(valid) < 3:
            rho, p = np.nan, np.nan
        else:
            if img_met in scores_img_metrics:
                valid[img_met] = - valid[img_met]
            rho, p = ebisuzaki_correlation_test(valid[img_met], valid[plant_err], n_surrogates=2000, metric='spearman')
        rho_row[plant_err] = float(rho) if np.isfinite(rho) else np.nan
        p_row[plant_err]   = float(p) if np.isfinite(p) else np.nan
    rho_rows.append(pd.Series(rho_row, name=img_met))
    p_rows.append(pd.Series(p_row, name=img_met))

CorrPearson = pd.DataFrame(rho_rows)
Pvalues      = pd.DataFrame(p_rows)

print("Corrélation (ρ Spearman):")
display(CorrPearson.round(3))

print("p-values:")
display(Pvalues.applymap(lambda x: round(x, 3) if pd.notna(x) else x))

# --- Heatmap simple (matplotlib) centrée sur 0 ---
plt.figure(figsize=(1.2*len(plant_rel_err)+3, 0.45*len(img_metrics)+3))
im = plt.imshow(CorrPearson.values, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, fraction=0.046, pad=0.04, label="Spearman r (img_metric vs plant_error)")
plt.yticks(range(len(img_metrics)), img_metrics)
plt.xticks(range(len(plant_rel_err)), plant_rel_err, rotation=45, ha="right")
plt.title("Correlation table (Spearman) between Segmentation metrics and Plant errors", pad=20)

# Annotations: r
for i in range(len(img_metrics)):
    for j in range(len(plant_rel_err)):
        rho = CorrPearson.iat[i, j]
        p = Pvalues.iat[i, j]
        txt = "" if np.isnan(rho) else f"{rho:.3f}\n(p={p:.3f})"
        plt.text(j, i, txt, ha='center', va='center', fontsize=8, color="black")
plt.tight_layout()
plt.savefig("corr_img_to_plant_error_spearman.png", transparent=True, dpi=600)
plt.show()

# Option: export CSV
#CorrPearson.to_csv("corr_img_to_plant_error_pearson.csv")
#Pvalues.to_csv("corr_img_to_plant_error_pearson_pvalues.csv")

## Per window analysis